# Projet Aegis
### Etude de stabilité financieres en bourse

Télechargement extraction et changements des type des fichiers de données

In [ ]:
import os
import re
import zipfile
from pathlib import Path
import requests
from bs4 import BeautifulSoup
import pandas as pd

non = 1

# Identifiant requis par l'API SEC
EN_TETES = {
    'User-Agent': 'ScriptPythonData chabot.arthur35@gmail.com',
    'Accept-Encoding': 'gzip, deflate'
}
# Dossier local où stocker les fichiers (créé dans le dossier du script)
DOSSIER_SORTIE = Path("./donnees_sec")

if non != 1:
    # Télechargement du json des ticker pour la correspondance cik-ticker
    reponse = requests.get("https://www.sec.gov/files/company_tickers.json",headers=EN_TETES, stream=True, timeout=60)
    reponse.raise_for_status()

    # Enregistre le contenu dans un fichier local
    with open(DOSSIER_SORTIE / "company_tickers.json", "w", encoding="utf-8") as f:
        f.write(reponse.text)

    def obtenir_urls():
        """Scrape la SEC pour récupérer les liens ZIP de 2010 à aujourd'hui."""
        url_sec = "https://www.sec.gov/dera/data/financial-statement-data-sets.html"
        print("Analyse de la page de la SEC...")

        reponse = requests.get(url_sec, headers=EN_TETES, timeout=30)
        reponse.raise_for_status()

        soup = BeautifulSoup(reponse.text, 'html.parser')
        urls_filtrees = []

        for balise_a in soup.find_all('a', href=True):
            lien = balise_a['href']
            if lien.lower().endswith('.zip'):
                match = re.search(r'(20\d{2})', lien)
                if match:
                    annee = int(match.group(1))
                    if annee >= 2010:
                        lien_complet = f"https://www.sec.gov{lien}" if lien.startswith('/') else lien
                        if lien_complet not in urls_filtrees:
                            urls_filtrees.append(lien_complet)

        print(f"{len(urls_filtrees)} archives ZIP détectées (2010+).")
        return urls_filtrees


    def traiter_donnees(urls):
        """Télécharge, isole dans un sous-dossier par trimestre et convertit en .paquets."""
        DOSSIER_SORTIE.mkdir(parents=True, exist_ok=True)
        fichiers_convertis = 0

        for i, url in enumerate(urls, 1):
            nom_zip = Path(url).name
            nom_dossier_trimestre = Path(nom_zip).stem
            
            chemin_zip = DOSSIER_SORTIE / nom_zip
            dossier_destination = DOSSIER_SORTIE / nom_dossier_trimestre

            print(f"\n[{i}/{len(urls)}] Téléchargement : {nom_zip}...")
            try:
                # 1. Téléchargement
                reponse = requests.get(url, headers=EN_TETES, stream=True, timeout=60)
                reponse.raise_for_status()

                with open(chemin_zip, "wb") as f:
                    for chunk in reponse.iter_content(chunk_size=65536):
                        f.write(chunk)

                # 2. Extraction dans un sous-dossier dédié pour éviter les écrasements
                dossier_destination.mkdir(parents=True, exist_ok=True)
                with zipfile.ZipFile(chemin_zip, 'r') as archive:
                    archive.extractall(dossier_destination)

                # 3. Conversion en .paquets
                for chemin_fichier in dossier_destination.rglob("*.txt"): # On ne cherche que les txt
                    nouveau_nom = chemin_fichier.with_suffix(".parquet")
                    
                    if not nouveau_nom.exists():
                        try:
                            # On charge le fichier texte en mémoire (séparé par des tabulations)
                            df = pd.read_csv(chemin_fichier, sep='\t', low_memory=False, on_bad_lines='skip')
                            
                            # On encode et sauvegarde au vrai format binaire compressé Parquet
                            df.to_parquet(nouveau_nom, engine='pyarrow')
                            # On supprime le .txt d'origine pour ne pas saturer ton disque dur
                            chemin_fichier.unlink()
                            
                            fichiers_convertis += 1
                            print(f"  -> Converti : {nouveau_nom.name}")
                        except Exception as e:
                            print(f"  -> Erreur de conversion sur {chemin_fichier.name}: {e}")

                # 4. Suppression du ZIP pour économiser votre espace disque
                chemin_zip.unlink()

            except Exception as err:
                print(f"Et merde {nom_zip} : {err}")
                continue

        print(f"\n Terminé !")
        print(f" Total de fichiers convertits en .parquets : {fichiers_convertis}")


    if __name__ == "__main__":
        liens = obtenir_urls()
        traiter_donnees(liens)
else:
    print("No download")

IndentationError: expected an indented block after 'with' statement on line 25 (1287644243.py, line 26)

Utilisation de DuckDB pour le stockage de toutes ces informations

In [23]:
import duckdb 
import time

con = duckdb.connect()

# Requête SQL pour croiser et pivoter les données de l'ensemble des fichiers
# L'étoile (*) dans read_parquet permet de lire tous les sous-dossiers d'un coup !
query = """
WITH donnees_filtrees AS (
    SELECT 
        s.cik,
        s.name AS company_name,
        s.filed AS filing_date,
        s.adsh,
        n.tag,
        n.value
    FROM read_parquet('donnees_sec/*/sub.parquet') AS s
    JOIN read_parquet('donnees_sec/*/num.parquet') AS n 
      ON s.adsh = n.adsh
    WHERE s.form = '10-K'  -- Uniquement les bilans annuels
      AND n.tag IN (
          'Assets', 
          'Liabilities', 
          'Revenues', 
          'SalesRevenueNet', 
          'OperatingIncomeLoss', 
          'NetIncomeLoss', 
          'NetCashProvidedByUsedInOperatingActivities'
      )
      -- Exclure les filiales (coreg vide ou nul = données de la société mère uniquement)
      AND (n.coreg IS NULL OR n.coreg = '')
      
      -- FILTRE VITAL SEC : On aligne la date comptable (ddate) avec la période du rapport (period)
      -- Cela évite de récupérer les chiffres des années précédentes déclarés à titre comparatif
      AND n.ddate = s.period 
)
-- ÉTAPE DE PIVOT : On transforme nos tags (lignes) en colonnes
SELECT 
    cik,
    company_name,
    filing_date,
    adsh,
    MAX(CASE WHEN tag = 'Assets' THEN value END) AS Assets,
    MAX(CASE WHEN tag = 'Liabilities' THEN value END) AS Liabilities,
    MAX(CASE WHEN tag IN ('Revenues', 'SalesRevenueNet') THEN value END) AS Revenues,
    MAX(CASE WHEN tag = 'OperatingIncomeLoss' THEN value END) AS OperatingIncome,
    MAX(CASE WHEN tag = 'NetIncomeLoss' THEN value END) AS NetIncome,
    MAX(CASE WHEN tag = 'NetCashProvidedByUsedInOperatingActivities' THEN value END) AS OperatingCash
FROM donnees_filtrees
GROUP BY cik, company_name, filing_date, adsh
"""

print("Exécution de la requête sur les fichiers Parquet (cela peut prendre 1 à 3 minutes)...")
start_time = time.time()

# Exécution par DuckDB et conversion immédiate en DataFrame Pandas
df_fondamentaux = con.execute(query).df()

print(f"Extraction terminée en {round(time.time() - start_time, 2)} secondes !")

# Suppression des éventuelles valeurs vides (entreprises avec des rapports incomplets)
df_propre = df_fondamentaux.dropna()

# Sauvegarde du résultat final compressé
fichier_sortie = "dataset_aegis_fondamentaux.parquet"
df_propre.to_parquet(fichier_sortie)

print(f"\nSuccès ! Fichier '{fichier_sortie}' créé avec succès.")
print(f"Total : {len(df_propre)} bilans annuels complets extraits sur 20 ans.")
print("\nAperçu des 5 premières lignes :")
print(df_propre.head())

Exécution de la requête sur les fichiers Parquet (cela peut prendre 1 à 3 minutes)...
Extraction terminée en 18.38 secondes !

Succès ! Fichier 'dataset_aegis_fondamentaux.parquet' créé avec succès.
Total : 21166 bilans annuels complets extraits sur 20 ans.

Aperçu des 5 premières lignes :
        cik          company_name  filing_date                  adsh  \
3     59440      VECTOR GROUP LTD     20110225  0000950123-11-018301   
6    749251           GARTNER INC     20110215  0000950123-11-014318   
7   1141391        MASTERCARD INC     20110224  0001193125-11-044721   
9     77360           PENTAIR INC     20110223  0000950123-11-016702   
10   831967  KINETIC CONCEPTS INC     20110301  0000831967-11-000010   

          Assets   Liabilities      Revenues  OperatingIncome     NetIncome  \
3   9.495950e+08  9.958290e+08  1.063289e+09     1.113130e+08  5.408400e+07   
6   1.285658e+09  1.098602e+09  1.288454e+09     1.492650e+08  9.628500e+07   
7   8.837000e+09  3.621000e+09  5.53900

Code qui se charge de la jointure cik et tickers car yfinance (info bourses) utilise le tickers.

In [25]:
import json
import pandas as pd
import duckdb
# Pandas "aplatit" le JSON complexe de la SEC en un beau tableau
chemin_json = 'donnees_sec/company_tickers.json'
with open(chemin_json, 'r') as f:
    donnees_json = json.load(f)
df_tickers = pd.DataFrame.from_dict(donnees_json, orient='index')

print("Fichier Parquet en cours de lecture et jointure...")

# Note : DuckDB peut lire directement les DataFrames Pandas qui sont dans ton script (ici 'df_tickers')
query = """
    SELECT
        d.*, 
        c.ticker AS Ticker
    FROM read_parquet('dataset_aegis_fondamentaux.parquet') AS d
    JOIN df_tickers AS c 
      ON CAST(d.cik AS INTEGER) = CAST(c.cik_str AS INTEGER)
"""

con = duckdb.connect()
df_final = con.execute(query).df()

# 4. Sauvegarde du nouveau tableau
fichier_sortie = "dataset_aegis_fondamentaux_avec_tickers.parquet"
df_final.to_parquet(fichier_sortie)

print(f"\nSuccès ! Jointure terminée.")
print(f"Lignes avant jointure : (dépend de ton ancien fichier)")
print(f"Lignes après jointure : {len(df_final)}")
print("\nAperçu des colonnes :", list(df_final.columns))
print(df_final[['company_name', 'cik', 'Ticker']].head())

Fichier Parquet en cours de lecture et jointure...

Succès ! Jointure terminée.
Lignes avant jointure : (dépend de ton ancien fichier)
Lignes après jointure : 12596

Aperçu des colonnes : ['cik', 'company_name', 'filing_date', 'adsh', 'Assets', 'Liabilities', 'Revenues', 'OperatingIncome', 'NetIncome', 'OperatingCash', '__index_level_0__', 'Ticker']
                        company_name      cik Ticker
0                        GARTNER INC   749251     IT
1                     MASTERCARD INC  1141391     MA
2                        PENTAIR INC    77360    PNR
3                        SNAP-ON INC    91440    SNA
4  WEIGHT WATCHERS INTERNATIONAL INC   105319     WW


In [29]:
import yfinance as yf

query = """
SELECT * FROM read_parquet('dataset_aegis_fondamentaux_avec_tickers.parquet')
"""

con = duckdb.connect()
df_final = con.execute(query).df()
print(df_final)

           cik                       company_name  filing_date  \
0       749251                        GARTNER INC     20110215   
1      1141391                     MASTERCARD INC     20110224   
2        77360                        PENTAIR INC     20110223   
3        91440                        SNAP-ON INC     20110217   
4       105319  WEIGHT WATCHERS INTERNATIONAL INC     20110302   
...        ...                                ...          ...   
12591  1045450                     EPR PROPERTIES     20250227   
12592  1042046       AMERICAN FINANCIAL GROUP INC     20140228   
12593  1232582      ASHFORD HOSPITALITY TRUST INC     20140303   
12594  1050446                  MICROSTRATEGY INC     20140212   
12595  1232582      ASHFORD HOSPITALITY TRUST INC     20140303   

                       adsh        Assets   Liabilities      Revenues  \
0      0000950123-11-014318  1.285658e+09  1.098602e+09  1.288454e+09   
1      0001193125-11-044721  8.837000e+09  3.621000e+09  5.53

In [27]:
import pandas as pd
import numpy as np
import yfinance as yf

def calculer_score_stabilite(serie_prix: pd.Series) -> dict:
    """
    Calcule le Rendement, le Max Drawdown et le Ratio de Calmar (Score de stabilité).
    
    Arguments:
    serie_prix (pd.Series) : Série temporelle des prix de l'action (index = dates, valeurs = prix).
    
    Retourne:
    Un dictionnaire contenant les 3 métriques. Si la série est vide ou invalide, retourne NaN.
    """
    
    # 1. Sécurité : Vérifier qu'on a bien au moins 10 jours de cotation dans l'année
    if len(serie_prix) < 10 or serie_prix.isnull().all():
        return {'rendement_annuel': np.nan, 'max_drawdown': np.nan, 'score_stabilite': np.nan}
    
    # On s'assure qu'il n'y a pas de trous (NaN) au milieu, on utilise le dernier prix connu
    serie_prix = serie_prix.ffill()
    
    # 2. Le Rendement Annuel
    prix_initial = serie_prix.iloc[0]
    prix_final = serie_prix.iloc[-1]
    
    # Si le prix initial est à 0 (bug de données), on évite une division par zéro
    if prix_initial <= 0:
        return {'rendement_annuel': np.nan, 'max_drawdown': np.nan, 'score_stabilite': np.nan}
        
    rendement = (prix_final - prix_initial) / prix_initial
    
    # 3. Le Max Drawdown (La pire chute)
    # On calcule d'abord le "sommet historique" (cummax) pour chaque jour
    sommets = serie_prix.cummax()
    
    # Le drawdown d'un jour précis = (Prix actuel - Sommet atteint jusque-là) / Sommet
    drawdowns = (serie_prix - sommets) / sommets
    
    # Le Max Drawdown est le chiffre le plus négatif de cette série (ex: -0.45 pour -45%)
    max_drawdown = drawdowns.min()
    
    # 4. Le Score de Stabilité (Ratio de Calmar modifié)
    # Règle de pénalité : si un actif ne baisse pas du tout (cas très rare), on évite une division par 0
    # On passe le drawdown en valeur absolue pour que le ratio ait du sens (Rendement positif / Chute positive)
    mdd_absolu = abs(max_drawdown)
    
    # On fixe un plancher au MDD (ex: 1%) pour ne pas avoir de scores infinis si une action n'a pas baissé
    mdd_effectif = max(mdd_absolu, 0.01) 
    
    score = rendement / mdd_effectif
    
    return {
        'rendement_annuel': rendement,
        'max_drawdown': max_drawdown,
        'score_stabilite': score
    }

# ==========================================
# EXEMPLE D'UTILISATION (Test grandeur nature)
# ==========================================
if __name__ == "__main__":
    print("Téléchargement des données d'Apple (AAPL) pour l'année 2023...")
    
    # Téléchargement via yfinance
    donnees_aapl = yf.download("AAPL", start="2023-01-01", end="2023-12-31", progress=False)
    
    # On extrait uniquement la colonne des prix de clôture (Close)
    # yfinance renvoie un DataFrame multi-index, on prend la colonne 'Close' pour 'AAPL'
    serie_aapl = donnees_aapl['Close']['AAPL'] if isinstance(donnees_aapl.columns, pd.MultiIndex) else donnees_aapl['Close']

    
    # Calcul
    resultat = calculer_score_stabilite(serie_aapl)
    
    print("\n--- RÉSULTATS APPLE 2023 ---")
    print(f"Rendement annuel : {resultat['rendement_annuel'] * 100:.2f} %")
    print(f"Pire chute (Max Drawdown) : {resultat['max_drawdown'] * 100:.2f} %")
    print(f"SCORE DE STABILITÉ : {resultat['score_stabilite']:.2f}")

Téléchargement des données d'Apple (AAPL) pour l'année 2023...

--- RÉSULTATS APPLE 2023 ---
Rendement annuel : 54.80 %
Pire chute (Max Drawdown) : -14.93 %
SCORE DE STABILITÉ : 3.67
